# Notebook 03 — Evaluation

End-to-end evaluation of the trained system:
- Solver accuracy on 20 benchmark equations
- Per-stage speed benchmark
- CNN per-class precision / recall / F1
- Full pipeline test on a synthetic equation image

**Pre-requisites:** model trained (`python main.py --train` or Notebook 02 completed).

In [ ]:
import sys, os, time
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pandas as pd
import config

## 1. Solver accuracy — 20 benchmark equations

In [ ]:
from demo.evaluation import SAMPLE_EQUATIONS, _check, test_on_sample_set

accuracy = test_on_sample_set()
print(f'\nSolver accuracy: {accuracy*100:.1f}%')

## 2. Speed benchmark — all pipeline stages

In [ ]:
from demo.evaluation import benchmark_speed

timings = benchmark_speed(n_runs=30)

stages = list(timings.keys())
avgs   = list(timings.values())

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(stages, avgs, color='steelblue', edgecolor='white')
ax.set_title('Average processing time per pipeline stage (ms)')
ax.set_ylabel('Time (ms)')
for bar, v in zip(bars, avgs):
    ax.text(bar.get_x() + bar.get_width() / 2, v + 0.1,
            f'{v:.1f}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig('../output/reports/03_speed_benchmark.png', dpi=120)
plt.show()

## 3. CNN per-class metrics (requires trained model)

In [ ]:
if not config.MODEL_PATH.exists():
    print('Model not found — skipping CNN evaluation.')
    print('Run: python main.py --train')
else:
    import tensorflow as tf
    from sklearn.metrics import classification_report
    from src.data_preparation import DataPreparator

    model = tf.keras.models.load_model(str(config.MODEL_PATH))
    dp = DataPreparator()
    _, _, X_test, _, _, y_test = dp.prepare()

    y_pred = model.predict(X_test, verbose=0).argmax(axis=1)
    y_true = y_test.argmax(axis=1)
    labels = [config.CLASS_MAP[i] for i in range(config.NUM_CLASSES)]

    report = classification_report(y_true, y_pred, target_names=labels, output_dict=True)
    df = pd.DataFrame(report).T.iloc[:config.NUM_CLASSES]
    df = df[['precision', 'recall', 'f1-score', 'support']].round(3)
    df.index.name = 'class'
    print(df.to_string())

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    for ax, metric in zip(axes, ['precision', 'recall', 'f1-score']):
        ax.bar(labels, df[metric], color='cornflowerblue', edgecolor='white')
        ax.set_title(metric.capitalize()); ax.set_ylim(0, 1.05)
        ax.set_xlabel('Class'); ax.axhline(0.9, color='red', linestyle='--', alpha=0.5)
    plt.suptitle('Per-class CNN metrics — test set', fontsize=11)
    plt.tight_layout()
    plt.savefig('../output/reports/03_per_class_metrics.png', dpi=120)
    plt.show()

## 4. Full pipeline test — synthetic equation image

In [ ]:
import cv2
from src.preprocessing import ImagePreprocessor
from src.segmentation  import Segmenter
from src.math_solver   import MathSolver
from src.step_formatter import StepFormatter

H, W = 200, 620
img  = np.full((H, W), 240, dtype=np.uint8)
font = cv2.FONT_HERSHEY_SIMPLEX
for glyph, cx in [('3', 40), ('+', 155), ('5', 270), ('=', 380), ('8', 490)]:
    cv2.putText(img, glyph, (cx, 140), font, 3.0, 25, 7, cv2.LINE_AA)

tmp_path = '../tests/output/_eval_input.png'
cv2.imwrite(tmp_path, img)

preprocessor = ImagePreprocessor()
normalized   = preprocessor.preprocess(tmp_path)
binary       = preprocessor.get_binary_image()

segmenter  = Segmenter()
characters = segmenter.segment(binary)

fig, axes = plt.subplots(1, 2, figsize=(12, 3))
axes[0].imshow(img, cmap='gray'); axes[0].set_title('Input image'); axes[0].axis('off')
axes[1].imshow(binary, cmap='gray'); axes[1].set_title(f'Binary — {len(characters)} chars detected'); axes[1].axis('off')
plt.tight_layout()
plt.savefig('../output/reports/03_pipeline_smoke.png', dpi=120)
plt.show()

print(f'Characters detected: {len(characters)}')
for c in characters:
    print(f"  bbox=({c['bbox']['x']},{c['bbox']['y']},{c['bbox']['width']},{c['bbox']['height']})  type={c['position_type']}")

## 5. Solver step-by-step demo

In [ ]:
from src.math_solver    import MathSolver
from src.step_formatter import StepFormatter

solver    = MathSolver()
formatter = StepFormatter()

demo_equations = [
    '2*x + 5 = 15',
    'x**2 - 5*x + 6 = 0',
    '25 + 37',
    '144/12',
]

for eq in demo_equations:
    result  = solver.solve_equation(eq)
    display = formatter.format_for_display(result)
    print(display)
    print()